# mcp

> the vault as MCP tools any client can drive

In [ ]:
#| default_exp mcp

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

The same `cmd` the CLI uses: a `Vault` method becomes a tool whose schema and description are the method's signature and docstring.

```json
{"mcpServers": {"vishalakshi": {"command": "vishalakshi-mcp",
                                "env": {"VISHALAKSHI_VAULT": "~/.vishalakshi/vault.db"}}}}
```


In [ ]:
#| export
import json, sys
from functools import wraps
from vishalakshi.cli import cmd, jsonable, vault

try: from mcp.server.mcpserver import MCPServer
except ImportError:
    try: from mcp.server.fastmcp import FastMCP as MCPServer
    except ImportError as e:
        raise ImportError('vishalakshi-mcp needs the `mcp` package: `pip install mcp`') from e


In [ ]:
#| export
TOOLS = ('stats search sections context ask read related toc map topic_tree sources document shelves elsewhere grab url '
         'web arxiv youtube github gh_file add_file add_dir add_tree note connect forget apis harvest watch watches '
         'poll unwatch index_code code_search symbol where_to_add grep federate categorize '
         'categorize_all doctypes of_type ner reshelf extract extract_all ask_doc '
         'pii mark_pii mark_not_pii mark_noisy mark_noisy_many marks '
         'suggest_noisy accept_noisy learn fit_noise use_noise').split()

mcp = MCPServer('vishalakshi', instructions=(
    'A personal research vault: web pages, papers, transcripts, files, code and notes in one '
    'searchable corpus. `context` is the main tool: it returns whole sections plus what they '
    'connect to, which is what you want before answering a question. `search` locates things, '
    '`read` pulls one section in full, `related` answers "what else reads like this". `grab` '
    'files anything you point it at; `note` writes your own conclusions back so they are searched '
    'alongside the sources. `add_tree` points at a directory and splits it: documents into the '
    'vault, source files into kosha. When kosha has indexed '
    'the repo, `context` and `ask` add code sections to what they retrieve on their own, so a '
    'question about the user\'s own system is answered from the source rather than from prose about '
    'it.\n\n'
    'For one document rather than the corpus: `document` returns the whole of it, `categorize` says '
    'what kind of thing it is (invoice, catalogue, contract, paper, …), `extract` pulls its fields '
    'out against a schema (a name like `invoice`, or a spec like `vendor:str, total:float` you '
    'make up on the spot), and `ask_doc` answers a question about that document with the rest of '
    'the vault as context, as prose or as a shape you name. `extract_all` does it across every '
    'document of one type, which is how a folder of invoices becomes a table.\n\n'
    'Judgement loop: `suggest_noisy` ranks junk and `mark_noisy` / `accept_noisy` exclude it from '
    'retrieval. `pii` says whether a document is private; `mark_pii` and `mark_not_pii` overrule '
    'the detector.'))

def as_tool(name:str):
    'Register one `Vault` method as an MCP tool, forcing its result through JSON.'
    f = cmd(name)
    g = wraps(f)(lambda **kw: json.loads(json.dumps(f(**kw), default=jsonable)))
    g.__signature__, g.__delwrap__, g.__annotations__ = f.__signature__, f.__delwrap__, f.__annotations__
    return mcp.tool(name=name)(g)

for _t in TOOLS: as_tool(_t)

def main():
    'Entry point for `vishalakshi-mcp`. stdio by default; `--http` for Streamable HTTP.'
    mcp.run(transport='streamable-http' if '--http' in sys.argv[1:] else 'stdio')

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/pydantic_settings/sources/utils.py:47: IncompleteFieldDefinitionWarning: Field 'lifespan' has an incomplete definition: its annotation contains an unresolved forward reference, so settings sources may fail to correctly resolve its value. Call `model_rebuild()` on the model where the field is defined, once all the referenced types are defined.
  warnings.warn(


## Try it

In [ ]:
tools = {t.name: t for t in await mcp.list_tools()}
len(tools), tools['context'].description[:120]

(47,
 'The retrieval an LLM should be handed: whole sections plus what they connect to. sections carry `text, breadcrumb, pages')

In [ ]:
def schema(t): return getattr(t, 'input_schema', None) or t.inputSchema   # renamed in mcp 2.0

test_eq(sorted(tools), sorted(TOOLS))
test_eq(schema(tools['search'])['properties']['limit']['default'], 10)
test_eq(schema(tools['search'])['required'], ['q'])
# the docments on the Vault method are what a client reads, so they have to survive the round trip
assert 'invoice' in tools['categorize'].description
test_eq(schema(tools['extract'])['required'], ['ref'])
test_eq(schema(tools['ask_doc'])['required'], ['ref', 'question'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()